In [ ]:

# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D11 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D11 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D11"
DOCUMENT_NAME = "Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24"
BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_RECORD_COUNT = 199

EXPECTED_CATEGORY_COUNTS = {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location",
]

IDENTITY_FIELDS = [
    "Category",
    "Topic",
    "Reporting Period",
]


PRIMARY_CORRECTNESS_FIELDS = [
    "Value",
    "Unit",
    "Source Location",
]

DIAGNOSTIC_FIELDS = [
    "Description",
]

EXPECTED_SOURCE_PAGES = {1, 2, 4, 5, 6, 8, 9, 10}

EXPECTED_QUALIFIED_VALUES = {
    "800+",
    "~100",
    "245+",
    "~1.85",
    "~4.5",
    "5 and counting",
}

OUTPUT_DIR = Path("outputs_D11_validation_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Identity fields:", IDENTITY_FIELDS)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)


In [ ]:

# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D11_reference_values.csv
#   2) D11_branch_B_parsed_extraction.json
#   3) D11_branch_B_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    name for name in uploaded_files
    if name.lower().endswith(".csv")
]

json_files = [
    name for name in uploaded_files
    if name.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one D11 Stage 1 reference CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B parsed "
        "extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]
PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None

for file_name in json_files:
    with open(file_name, "r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structurally_evaluable" in obj
        and "valid_json" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D11 Branch B parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D11 Branch B technical diagnostics."
    )

print("Reference:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Technical diagnostics:", TECHNICAL_DIAGNOSTICS_FILE)


In [ ]:

# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

reference_df = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=True,
    encoding="utf-8-sig",
)

reference_df = reference_df.where(
    pd.notna(reference_df),
    None,
)

def restore_reference_value(value):
    if value is None or pd.isna(value):
        return None

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return value

    text = str(value).strip()

    if (
        text.startswith("~")
        or text.endswith("+")
        or "and counting" in text.casefold()
    ):
        return text

    if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
        number = float(text)
        return int(number) if number.is_integer() else number

    return text

reference_df["Value"] = reference_df["Value"].map(
    restore_reference_value
)

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    parsed_extraction = json.load(f)

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    technical_diagnostics = json.load(f)

for artefact_name, artefact in {
    "parsed extraction": parsed_extraction,
    "technical diagnostics": technical_diagnostics,
}.items():
    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )

extracted_df = pd.DataFrame(
    parsed_extraction["records"]
)

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256": sha256_file(PARSED_EXTRACTION_FILE),
    "technical_diagnostics_file": TECHNICAL_DIAGNOSTICS_FILE,
    "technical_diagnostics_sha256":
        sha256_file(TECHNICAL_DIAGNOSTICS_FILE),
}

print("Reference shape:", reference_df.shape)
print("Extraction shape:", extracted_df.shape)


In [ ]:

# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

record_schema_valid = bool(
    technical_diagnostics.get(
        "record_schema_valid",
        structurally_evaluable
    )
)

field_types_valid = bool(
    technical_diagnostics.get(
        "field_types_valid",
        structurally_evaluable
    )
)

schema_validity = bool(
    structurally_evaluable
    and record_schema_valid
    and field_types_valid
)

schema_diagnostics = {
    "valid_json": bool(
        technical_diagnostics.get("valid_json", False)
    ),
    "record_schema_valid": record_schema_valid,
    "field_types_valid": field_types_valid,
    "structurally_evaluable": structurally_evaluable,
    "schema_validity": schema_validity,
}

if not structurally_evaluable:
    raise ValueError(
        "D11 Branch B output is not structurally evaluable. "
        "Content-level validation cannot proceed."
    )

print(json.dumps(
    schema_diagnostics,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 5. Confirm current Stage 1 D11 reference semantics
# ============================================================

def extract_page(value):
    if value is None:
        return None

    match = re.fullmatch(
        r"PDF page\s+(\d+)",
        str(value).strip(),
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    return int(
        match.group(1)
    )


observed_source_pages = {
    page
    for page in (
        reference_df["Source Location"]
        .map(extract_page)
    )
    if page is not None
}

observed_qualified_values = {
    str(value)
    for value in reference_df["Value"]
    if str(value) in EXPECTED_QUALIFIED_VALUES
}

identity_duplicate_count = int(
    reference_df.duplicated(
        subset=IDENTITY_FIELDS,
        keep=False,
    ).sum()
)

timeline_year_units_valid = all([
    reference_df.loc[
        (reference_df["Category"] == "Corporate timeline")
        & (reference_df["Topic"] == "Established"),
        "Unit",
    ].eq("year").all(),

    reference_df.loc[
        (reference_df["Category"] == "Corporate timeline")
        & (reference_df["Topic"] == "Public listing"),
        "Unit",
    ].eq("year").all(),
])

reference_semantic_checks = {
    "reference_schema_exact":
        bool(reference_schema_exact),

    "reference_types_valid":
        bool(reference_types_valid),

    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "reference_identity_unique":
        identity_duplicate_count == 0,

    "source_page_coverage_valid":
        observed_source_pages
        == EXPECTED_SOURCE_PAGES,

    "qualified_values_preserved":
        observed_qualified_values
        == EXPECTED_QUALIFIED_VALUES,

    "timeline_year_units_valid":
        bool(
            timeline_year_units_valid
        ),

    "no_reference_records_from_divider_pages":
        not bool(
            observed_source_pages
            & {3, 7}
        ),
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)

print(
    json.dumps(
        reference_semantic_checks,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "Current D11 reference semantics valid:",
    reference_semantics_valid,
)

if not reference_semantics_valid:
    raise AssertionError(
        "The supplied D11 reference does not match the "
        "current frozen Stage 1 D11 reference semantics."
    )


In [ ]:
# ============================================================
# 6. Comparison-only normalisation
# ============================================================

def normalise_text(value):
    if value is None:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = (
        text
        .replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
        .replace("—", "-")
        .replace("–", "-")
        .replace("‑", "-")
        .replace("’", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text.casefold()


def normalise_topic(value):
    return normalise_text(
        value
    )


PERIOD_EQUIVALENCE = {
    "q4fy24": "q4 fy24",
    "q4fy23": "q4 fy23",
    "q3fy24": "q3 fy24",
    "q4 & fy24": "q4 & fy24",
    "q4 and fy24": "q4 & fy24",
    "31 march 2024": "2024-03-31",
    "march 31, 2024": "2024-03-31",
    "march 31 2024": "2024-03-31",
    "2024-03-31": "2024-03-31",
}


def canonical_period(value):
    text = normalise_text(
        value
    )

    return PERIOD_EQUIVALENCE.get(
        text,
        text,
    )


UNIT_EQUIVALENCE = {
    "text": "text",
    "rs. mn": "rs mn",
    "rs mn": "rs mn",
    "₹ mn": "rs mn",
    "₹ in mn": "rs mn",
    "rs. crores": "rs crores",
    "rs crores": "rs crores",
    "rs. per share": "rs per share",
    "rs per share": "rs per share",
    "rs.": "rs",
    "rs": "rs",
    "%": "percent",
    "percentage": "percent",
    "percent": "percent",
    "store": "stores",
    "stores": "stores",
    "distributor": "distributors",
    "distributors": "distributors",
    "mn meter": "mn meters",
    "mn meters": "mn meters",
    "mn piece": "mn pieces",
    "mn pieces": "mn pieces",
    "mn customer": "mn customers",
    "mn customers": "mn customers",
    "l sq ft": "l sqft",
    "l sqft": "l sqft",
    "year": "year",
}


def canonical_unit(value):
    text = normalise_text(
        value
    )

    return UNIT_EQUIVALENCE.get(
        text,
        text,
    )


def parse_numeric(value):
    if isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if not isinstance(value, str):
        return None

    text = value.strip().replace(
        ",",
        "",
    )

    if re.fullmatch(
        r"-?\d+(?:\.\d+)?",
        text,
    ):
        try:
            return float(text)
        except ValueError:
            return None

    return None


def has_source_qualifier(value):
    if not isinstance(value, str):
        return False

    text = value.strip().casefold()

    return (
        text.startswith("~")
        or text.endswith("+")
        or "and counting" in text
    )


def value_equal(reference_value, extracted_value):
    if reference_value is None and extracted_value is None:
        return True

    if (
        has_source_qualifier(reference_value)
        or has_source_qualifier(extracted_value)
    ):
        return (
            normalise_text(reference_value)
            == normalise_text(extracted_value)
        )

    ref_numeric = parse_numeric(
        reference_value
    )

    ext_numeric = parse_numeric(
        extracted_value
    )

    if (
        ref_numeric is not None
        and ext_numeric is not None
    ):
        return math.isclose(
            ref_numeric,
            ext_numeric,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


def unit_equal(reference_value, extracted_value):
    return (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    )


def source_location_equal(reference_value, extracted_value):
    return (
        extract_page(reference_value)
        == extract_page(extracted_value)
        and extract_page(reference_value)
        is not None
    )


def description_exact(reference_value, extracted_value):
    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


In [ ]:
# ============================================================
# 7. Controlled one-to-one identity alignment
# ============================================================


def lexical_similarity(left, right):
    left_norm = normalise_text(left)
    right_norm = normalise_text(right)

    if left_norm == right_norm:
        return 1.0

    if not left_norm or not right_norm:
        return 0.0

    return SequenceMatcher(
        None,
        left_norm,
        right_norm,
    ).ratio()


def topic_similarity(left, right):
    left_norm = normalise_topic(left)
    right_norm = normalise_topic(right)

    if left_norm == right_norm:
        return 1.0

    if not left_norm or not right_norm:
        return 0.0

    return SequenceMatcher(
        None,
        left_norm,
        right_norm,
    ).ratio()


def period_similarity(left, right):
    left_period = canonical_period(left)
    right_period = canonical_period(right)

    if left_period == right_period:
        return 1.0

    if left_period is None or right_period is None:
        return 0.0

    return lexical_similarity(
        left_period,
        right_period,
    )


# ------------------------------------------------------------
# Frozen D11 alignment parameters from revised Branch A
# ------------------------------------------------------------

ALIGNMENT_WEIGHTS = {
    "topic": 0.75,
    "reporting_period": 0.25,
}

ALIGNMENT_SCORE_THRESHOLD = 0.60


def alignment_score(
    reference_record,
    extracted_record,
):

    if (
        normalise_text(
            reference_record["Category"]
        )
        !=
        normalise_text(
            extracted_record["Category"]
        )
    ):
        return 0.0

    topic_score = topic_similarity(
        reference_record["Topic"],
        extracted_record["Topic"],
    )

    period_score = period_similarity(
        reference_record["Reporting Period"],
        extracted_record["Reporting Period"],
    )

    return (
        ALIGNMENT_WEIGHTS["topic"]
        * topic_score
        +
        ALIGNMENT_WEIGHTS["reporting_period"]
        * period_score
    )


reference_records = (
    reference_df
    .to_dict("records")
)

extraction_records = (
    extracted_df
    .to_dict("records")
)


matches = []

matched_reference_indices = set()
matched_extraction_indices = set()

alignment_diagnostics = []


# ------------------------------------------------------------
# Step 1 — strict one-to-one identities
# ------------------------------------------------------------

def strict_identity_key(record):
    return (
        normalise_text(
            record["Category"]
        ),
        normalise_topic(
            record["Topic"]
        ),
        canonical_period(
            record["Reporting Period"]
        ),
    )


reference_identity_map = defaultdict(list)
extraction_identity_map = defaultdict(list)


for index, record in enumerate(
    reference_records
):
    reference_identity_map[
        strict_identity_key(record)
    ].append(index)


for index, record in enumerate(
    extraction_records
):
    extraction_identity_map[
        strict_identity_key(record)
    ].append(index)


reference_duplicate_identity_count = sum(
    len(indices) - 1
    for indices
    in reference_identity_map.values()
    if len(indices) > 1
)

extraction_duplicate_identity_count = sum(
    len(indices) - 1
    for indices
    in extraction_identity_map.values()
    if len(indices) > 1
)


for key in sorted(
    set(reference_identity_map)
    & set(extraction_identity_map)
):

    ref_indices = reference_identity_map[key]
    ext_indices = extraction_identity_map[key]

    if (
        len(ref_indices) == 1
        and len(ext_indices) == 1
    ):
        ref_index = ref_indices[0]
        ext_index = ext_indices[0]

        matches.append({
            "Reference Index":
                ref_index,

            "Extraction Index":
                ext_index,

            "Alignment Rule":
                "Strict Category + Topic + Reporting Period",

            "Alignment Score":
                1.0,

            "Topic Similarity":
                1.0,

            "Reporting Period Similarity":
                1.0,
        })

        matched_reference_indices.add(
            ref_index
        )

        matched_extraction_indices.add(
            ext_index
        )


# ------------------------------------------------------------
# Step 2 — controlled one-to-one fallback within Category
# ------------------------------------------------------------

categories = sorted(
    {
        normalise_text(
            record["Category"]
        )
        for record
        in reference_records
    }
    |
    {
        normalise_text(
            record["Category"]
        )
        for record
        in extraction_records
    }
)


for category in categories:

    remaining_reference_indices = [
        index
        for index, record
        in enumerate(reference_records)
        if (
            index
            not in matched_reference_indices
            and normalise_text(
                record["Category"]
            ) == category
        )
    ]

    remaining_extraction_indices = [
        index
        for index, record
        in enumerate(extraction_records)
        if (
            index
            not in matched_extraction_indices
            and normalise_text(
                record["Category"]
            ) == category
        )
    ]

    if (
        not remaining_reference_indices
        or not remaining_extraction_indices
    ):
        continue


    score_matrix = np.zeros(
        (
            len(
                remaining_reference_indices
            ),
            len(
                remaining_extraction_indices
            ),
        ),
        dtype=float,
    )


    for row_index, ref_index in enumerate(
        remaining_reference_indices
    ):

        reference_record = (
            reference_records[
                ref_index
            ]
        )

        for column_index, ext_index in enumerate(
            remaining_extraction_indices
        ):

            extracted_record = (
                extraction_records[
                    ext_index
                ]
            )

            score_matrix[
                row_index,
                column_index
            ] = alignment_score(
                reference_record,
                extracted_record,
            )


    # Hungarian assignment maximises total identity score.
    row_indices, column_indices = (
        linear_sum_assignment(
            -score_matrix
        )
    )


    for row_index, column_index in zip(
        row_indices,
        column_indices,
    ):

        score = float(
            score_matrix[
                row_index,
                column_index
            ]
        )

        if (
            score
            < ALIGNMENT_SCORE_THRESHOLD
        ):
            continue


        ref_index = (
            remaining_reference_indices[
                row_index
            ]
        )

        ext_index = (
            remaining_extraction_indices[
                column_index
            ]
        )

        reference_record = (
            reference_records[
                ref_index
            ]
        )

        extracted_record = (
            extraction_records[
                ext_index
            ]
        )


        topic_score = topic_similarity(
            reference_record["Topic"],
            extracted_record["Topic"],
        )

        period_score = period_similarity(
            reference_record[
                "Reporting Period"
            ],
            extracted_record[
                "Reporting Period"
            ],
        )


        match = {
            "Reference Index":
                ref_index,

            "Extraction Index":
                ext_index,

            "Alignment Rule":
                "Controlled Category-blocked fallback",

            "Alignment Score":
                score,

            "Topic Similarity":
                topic_score,

            "Reporting Period Similarity":
                period_score,
        }


        matches.append(
            match
        )

        alignment_diagnostics.append(
            match.copy()
        )

        matched_reference_indices.add(
            ref_index
        )

        matched_extraction_indices.add(
            ext_index
        )


# ------------------------------------------------------------
# Final unmatched observations
# ------------------------------------------------------------

missing_reference_indices = sorted(
    set(
        range(
            len(reference_records)
        )
    )
    - matched_reference_indices
)

unsupported_extraction_indices = sorted(
    set(
        range(
            len(extraction_records)
        )
    )
    - matched_extraction_indices
)


strict_alignment_count = sum(
    match[
        "Alignment Rule"
    ].startswith("Strict")
    for match in matches
)

fallback_alignment_count = sum(
    match[
        "Alignment Rule"
    ].startswith("Controlled")
    for match in matches
)


print(
    "Aligned records:",
    len(matches),
)

print(
    "Strict exact alignments:",
    strict_alignment_count,
)

print(
    "Controlled fallback alignments:",
    fallback_alignment_count,
)

print(
    "Missing reference records:",
    len(
        missing_reference_indices
    ),
)

print(
    "Unsupported/unmatched extraction records:",
    len(
        unsupported_extraction_indices
    ),
)

print(
    "Reference duplicate identity count:",
    reference_duplicate_identity_count,
)

print(
    "Extraction duplicate identity count:",
    extraction_duplicate_identity_count,
)

print(
    "Alignment threshold:",
    ALIGNMENT_SCORE_THRESHOLD,
)

print(
    "Alignment weights:",
    ALIGNMENT_WEIGHTS,
)


In [ ]:
# ============================================================
# 8. Field comparison
# ============================================================

comparison_rows = []

for match in matches:

    reference_record = reference_records[
        match["Reference Index"]
    ]

    extracted_record = extraction_records[
        match["Extraction Index"]
    ]

    row = {
        "Reference Index":
            match["Reference Index"],

        "Extraction Index":
            match["Extraction Index"],

        "Alignment Rule":
            match["Alignment Rule"],

        "Alignment Score":
            match.get(
                "Alignment Score",
                1.0,
            ),

        "Topic Similarity":
            match.get(
                "Topic Similarity",
                1.0,
            ),

        "Reporting Period Similarity":
            match.get(
                "Reporting Period Similarity",
                1.0,
            ),
    }

    correctness = {
        "Category":
            normalise_text(
                reference_record["Category"]
            )
            == normalise_text(
                extracted_record["Category"]
            ),

        "Topic":
            normalise_topic(
                reference_record["Topic"]
            )
            == normalise_topic(
                extracted_record["Topic"]
            ),

        "Description":
            description_exact(
                reference_record["Description"],
                extracted_record["Description"],
            ),

        "Value":
            value_equal(
                reference_record["Value"],
                extracted_record["Value"],
            ),

        "Unit":
            unit_equal(
                reference_record["Unit"],
                extracted_record["Unit"],
            ),

        "Reporting Period":
            canonical_period(
                reference_record["Reporting Period"]
            )
            == canonical_period(
                extracted_record["Reporting Period"]
            ),

        "Source Location":
            source_location_equal(
                reference_record["Source Location"],
                extracted_record["Source Location"],
            ),
    }

    for field in FIELDS:

        row[
            f"Reference {field}"
        ] = reference_record[field]

        row[
            f"Extracted {field}"
        ] = extracted_record[field]

        row[
            f"{field} Correct"
        ] = bool(
            correctness[field]
        )

    row[
        "Fully Correct Primary Record"
    ] = all(
        correctness[field]
        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )

    comparison_rows.append(
        row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


missing_records_df = (
    reference_df
    .iloc[
        missing_reference_indices
    ]
    .copy()
)

unsupported_records_df = (
    extracted_df
    .iloc[
        unsupported_extraction_indices
    ]
    .copy()
)

discrepant_records_df = (
    comparison_df.loc[
        ~comparison_df[
            "Fully Correct Primary Record"
        ]
    ]
    .copy()
)


print(
    "Fully correct primary records:",
    int(
        comparison_df[
            "Fully Correct Primary Record"
        ].sum()
    ),
)

print(
    "Primary discrepant records:",
    len(
        discrepant_records_df
    ),
)


In [ ]:

# ============================================================
# 9. Metrics
# ============================================================

aligned_records = len(comparison_df)
fully_correct_records = int(
    comparison_df["Fully Correct Primary Record"].sum()
)
discrepant_records = aligned_records - fully_correct_records
missing_records = len(missing_reference_indices)
unsupported_records = len(unsupported_extraction_indices)

N_REF = int(len(reference_df))
N_EXT = int(len(extracted_df))

completeness = (
    aligned_records / N_REF
    if N_REF else 0.0
)

missing_rate = (
    missing_records / N_REF
    if N_REF else 0.0
)

record_precision_exact = (
    fully_correct_records / N_EXT
    if N_EXT else 0.0
)

record_recall_exact = (
    fully_correct_records / N_REF
    if N_REF else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if record_precision_exact + record_recall_exact
    else 0.0
)

unsupported_rate = (
    unsupported_records / N_EXT
    if N_EXT else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_records / aligned_records
    if aligned_records else 0.0
)

primary_field_accuracy_among_aligned = {
    field: (
        float(comparison_df[f"{field} Correct"].mean())
        if aligned_records else 0.0
    )
    for field in PRIMARY_CORRECTNESS_FIELDS
}

diagnostic_field_accuracy = {
    "Description exact": (
        float(comparison_df["Description Correct"].mean())
        if aligned_records else 0.0
    )
}

correct_primary_field_instances = int(
    sum(
        comparison_df[f"{field} Correct"].sum()
        for field in PRIMARY_CORRECTNESS_FIELDS
    )
)

expected_primary_field_instances = int(
    N_REF * len(PRIMARY_CORRECTNESS_FIELDS)
)

field_accuracy = (
    correct_primary_field_instances
    / expected_primary_field_instances
    if expected_primary_field_instances
    else 0.0
)

category_metrics = {}

for category, expected_count in EXPECTED_CATEGORY_COUNTS.items():
    ref_subset = reference_df[
        reference_df["Category"] == category
    ]

    ext_subset = extracted_df[
        extracted_df["Category"] == category
    ]

    aligned_subset = comparison_df[
        comparison_df["Reference Category"] == category
    ]

    correct_count = int(
        aligned_subset["Fully Correct Primary Record"].sum()
    )

    precision = (
        correct_count / len(ext_subset)
        if len(ext_subset) else 0.0
    )

    recall = (
        correct_count / len(ref_subset)
        if len(ref_subset) else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    category_metrics[category] = {
        "expected_records": int(expected_count),
        "extracted_records": int(len(ext_subset)),
        "aligned_records": int(len(aligned_subset)),
        "fully_correct_records": correct_count,
        "discrepant_records":
            int(len(aligned_subset) - correct_count),
        "completeness": float(
            len(aligned_subset) / expected_count
            if expected_count else 0.0
        ),
        "record_precision_exact": float(precision),
        "record_recall_exact": float(recall),
        "record_f1_exact": float(f1),
    }

print("Reference records:", N_REF)
print("Extracted records:", N_EXT)
print("Aligned records:", aligned_records)
print("Fully correct records:", fully_correct_records)
print("Discrepant records:", discrepant_records)
print("Missing records:", missing_records)
print("Unsupported/unmatched records:", unsupported_records)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print("Field accuracy:", round(field_accuracy, 4))
print("Schema valid:", schema_validity)


In [ ]:

# ============================================================
# 10. Build final Branch B validation summary
# ============================================================

field_accuracy_among_aligned = {
    field: (
        float(comparison_df[f"{field} Correct"].mean())
        if aligned_records else 0.0
    )
    for field in FIELDS
}

summary = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": int(aligned_records),
    "fully_correct_records": int(fully_correct_records),
    "discrepant_records": int(discrepant_records),
    "missing_records": int(missing_records),
    "unsupported_extracted_records": int(unsupported_records),

    "completeness": round(completeness, 4),
    "missing_rate": round(missing_rate, 4),
    "record_precision_exact": round(record_precision_exact, 4),
    "record_recall_exact": round(record_recall_exact, 4),
    "record_f1_exact": round(record_f1_exact, 4),
    "unsupported_rate": round(unsupported_rate, 4),
    "discrepancy_rate_among_aligned":
        round(discrepancy_rate_among_aligned, 4),
    "field_accuracy": round(field_accuracy, 4),

    "field_accuracy_among_aligned": {
        key: round(value, 4)
        for key, value in field_accuracy_among_aligned.items()
    },

    "schema_validity": schema_validity,
    "schema_diagnostics": schema_diagnostics,
    "structurally_evaluable": structurally_evaluable,

    "identity_fields": IDENTITY_FIELDS,
    "primary_correctness_fields": PRIMARY_CORRECTNESS_FIELDS,
    "diagnostic_fields": DIAGNOSTIC_FIELDS,

    "matching_rules": {
        "identity_fields": IDENTITY_FIELDS,
        "matching_method":
            "Deterministic one-to-one alignment frozen from Validation A",
        "value_used_for_alignment": False,
        "unit_used_for_alignment": False,
        "description_used_for_formal_correctness": False,
    },

    "comparison_rules_frozen_from_branch_A": True,

    "normalisation_note": (
        "Controlled deterministic normalisation was applied only "
        "to comparison copies; the preserved Branch B extraction "
        "was not modified."
    ),

    "reference_semantics_valid": bool(reference_semantics_valid),
    "reference_semantic_checks": reference_semantic_checks,
    "category_metrics": category_metrics,
    "input_provenance": input_provenance,
}

print(json.dumps(
    summary,
    indent=2,
    ensure_ascii=False
))


In [ ]:

# ============================================================
# 11. Validation integrity checks
# ============================================================

assert reference_semantics_valid

assert (
    aligned_records + missing_records
    == N_REF
)

assert (
    aligned_records + unsupported_records
    == N_EXT
)

assert (
    fully_correct_records + discrepant_records
    == aligned_records
)

for metric_name, metric_value in {
    "completeness": completeness,
    "missing_rate": missing_rate,
    "record_precision_exact": record_precision_exact,
    "record_recall_exact": record_recall_exact,
    "record_f1_exact": record_f1_exact,
    "unsupported_rate": unsupported_rate,
    "discrepancy_rate": discrepancy_rate_among_aligned,
    "field_accuracy": field_accuracy,
}.items():
    assert 0.0 <= metric_value <= 1.0, (
        f"Invalid {metric_name}: {metric_value}"
    )

print("Validation integrity checks passed.")


In [ ]:

# ============================================================
# 12. Export and download validation artefacts
# ============================================================

fully_correct_records_df = comparison_df.loc[
    comparison_df["Fully Correct Primary Record"]
].copy()

field_validation_df = pd.DataFrame([
    {
        "Field": field,
        "Role": (
            "Primary correctness"
            if field in PRIMARY_CORRECTNESS_FIELDS
            else (
                "Diagnostic"
                if field in DIAGNOSTIC_FIELDS
                else "Identity"
            )
        ),
        "Correct": int(
            comparison_df[f"{field} Correct"].sum()
        ),
        "Aligned Records": int(aligned_records),
        "Accuracy Among Aligned": (
            float(comparison_df[f"{field} Correct"].mean())
            if aligned_records else 0.0
        ),
        "Overall Accuracy Against Reference": (
            float(
                comparison_df[f"{field} Correct"].sum()
                / N_REF
            )
            if N_REF else 0.0
        ),
    }
    for field in FIELDS
])

category_metrics_df = pd.DataFrame([
    {"Category": category, **metrics}
    for category, metrics in category_metrics.items()
])

SUMMARY_PATH = (
    OUTPUT_DIR / "D11_branch_B_validation_summary.json"
)
DETAILED_PATH = (
    OUTPUT_DIR / "D11_branch_B_validation_detailed.csv"
)
FULLY_CORRECT_PATH = (
    OUTPUT_DIR / "D11_branch_B_fully_correct_records.csv"
)
DISCREPANT_PATH = (
    OUTPUT_DIR / "D11_branch_B_discrepant_records.csv"
)
MISSING_PATH = (
    OUTPUT_DIR / "D11_branch_B_missing_records.csv"
)
UNSUPPORTED_PATH = (
    OUTPUT_DIR / "D11_branch_B_unsupported_records.csv"
)
FIELD_VALIDATION_PATH = (
    OUTPUT_DIR / "D11_branch_B_field_validation.csv"
)
CATEGORY_METRICS_PATH = (
    OUTPUT_DIR / "D11_branch_B_category_metrics.csv"
)
ALIGNMENT_ISSUES_PATH = (
    OUTPUT_DIR / "D11_branch_B_alignment_issues.json"
)

SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig",
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig",
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig",
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig",
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

ALIGNMENT_ISSUES_PATH.write_text(
    json.dumps(
        {
            "missing_reference_indices":
                missing_reference_indices,
            "unsupported_extraction_indices":
                unsupported_extraction_indices,
            "alignment_diagnostics":
                alignment_diagnostics,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

generated_outputs = [
    SUMMARY_PATH,
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    FIELD_VALIDATION_PATH,
    CATEGORY_METRICS_PATH,
    ALIGNMENT_ISSUES_PATH,
]

print("Generated D11 Validation B outputs:")

for path in generated_outputs:
    print("-", path.name)

for path in generated_outputs:
    files.download(path)
